# RQ III: Agent ablations


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq3')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

## Sequential agent ablations


In [ ]:
from reported_runs.analysis.sections_rq1_3 import diagnostic_tables, horizon_counts, reflection_totals, arm_rank_statistics, profile_review_details
from reported_runs.analysis.agent_ablation_views import ABLATION_COMPARISON_ORDER, label_ablation_comparisons, ablation_effect_ranges
agents = trajectory.loc[trajectory.study.eq('movielens_feedback_a300_e804') & trajectory.model.eq('agentic_recommender') & trajectory.stage.str.startswith(('L2', 'L3'))].copy()
ablations = label_ablation_comparisons(effect_summaries.loc[effect_summaries.study.eq('movielens_feedback_a300_e804') & effect_summaries.kind.eq('ablation')])
report.table(label_ablation_comparisons(results['comparison_plan'].loc[results['comparison_plan'].kind.eq('ablation')]), 'rq3_ablation_controls', "")

## Recorded memory use and forced actions


In [ ]:
from reported_runs.analysis.agent_usage import load_agent_usage
from reported_runs.analysis.sections_rq1_3 import agent_usage_summary
usage_sources = sources.loc[sources.model.eq('agentic_recommender') & ((sources.study.eq('movielens_feedback_a300_e804') & sources.stage.str.startswith(('L2', 'L3'))) | sources.study.eq('movielens_sampling_outcome_a150_r150_e804'))]
usage = load_agent_usage(usage_sources, repo, repo / 'reported_runs/analysis/cache/agent_usage')
canonical_usage = usage.loc[usage.arm.eq('canonical')]
usage_columns = ['Protocol', 'Presentation arm', 'Runs', 'Decisions', 'Memory calls', 'Forced actions']
report.table(agent_usage_summary(canonical_usage), 'rq3_memory_forced_action_summary', "", columns=usage_columns, source_frame=canonical_usage)
report.table(agent_usage_summary(canonical_usage, by_condition=True), 'rq3_memory_forced_actions_by_condition', "", columns=['Protocol', 'Seed', 'Condition', 'Decisions', 'Memory calls', 'Forced actions'], source_frame=canonical_usage)
probe_usage = usage.loc[usage.arm.ne('canonical')]
report.table(agent_usage_summary(probe_usage), 'rq3_probe_memory_forced_actions', "", columns=usage_columns, source_frame=probe_usage)

## Bias trajectories and successive configuration effects


In [ ]:
figure = effect_grid(ablations, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], title='Agent ablations and decision-input noise check', scales=scales, value='value', facet='window', category_order=ABLATION_COMPARISON_ORDER)
report.figure(figure, ablations, 'rq3_ablation_effects', "", recommended=False, why="")
effect_ranges = ablation_effect_ranges(ablations, ['head_click_share', 'head_exposure', 'weighted_head_exposure'])
report.table(effect_ranges, 'rq3_effect_ranges_and_noise_reference', "", columns=['comparison', 'metric', 'window', 'n_seed_blocks', 'min_effect', 'max_effect', 'reference_min', 'reference_max'], source_frame=ablations)
figure = trajectory_grid(agents, ['head_click_share', 'head_exposure', 'weighted_head_exposure'], title='Agent configurations: canonical trajectories', scales=scales, series='condition_label')
report.figure(figure, agents, 'rq3_agent_trajectories', "")
diagnostic_tables(report, agents, {'head_outcomes': ['head_click_share', 'head_exposure', 'weighted_head_exposure'], 'selection': ['click_exposure_gap', 'weighted_click_exposure_gap']}, prefix='rq3_agents', context="")

## Click behaviour, coverage, and repeated responses


In [ ]:
diagnostic_tables(report, agents, {'click_behaviour': ['click_rate', 'no_click_rate', 'invalid_rate', 'valid_clicks'], 'coverage': ['exposure_coverage', 'click_coverage'], 'concentration': ['exposure_gini', 'click_gini'], 'rank_statistics': ['mean_clicked_rank', 'first_position_click_share']}, prefix='rq3_agents', context="")
agent_probes = label_conditions(results['probes'].loc[results['probes'].study.eq('movielens_feedback_a300_e804') & results['probes'].model.eq('agentic_recommender') & results['probes'].stage.str.startswith(('L2', 'L3'))])
diagnostic_tables(report, agent_probes, {'agreement': ['replica_agreement', 'shuffled_agreement', 'agreement_sensitivity'], 'actions': ['replica_action_agreement', 'replica_action_disagreement', 'shuffled_action_agreement', 'shuffled_action_disagreement'], 'eligibility': ['triple_valid_clicks', 'replica_valid_pairs', 'shuffled_valid_pairs'], 'excluded_pairs': ['replica_excluded_non_valid_pairs', 'shuffled_excluded_non_valid_pairs']}, prefix='rq3_agents', context="")

In [ ]:
agent_ranks = results['ranks'].loc[results['ranks'].study.eq('movielens_feedback_a300_e804') & results['ranks'].model.eq('agentic_recommender') & results['ranks'].stage.str.startswith(('L2', 'L3'))]
for condition, selected in agent_ranks.groupby('condition_id', sort=False):
    rank_counts = rank_summary(selected)
    figure = rank_grid(rank_counts, title='Agent clicked ranks: ' + label_conditions(selected).condition_label.iloc[0])
    report.figure(figure, rank_counts, 'rq3_ranks_' + condition.lower().replace('__', '_'), "")
    report.table(rank_counts, 'rq3_rank_counts_' + condition.lower().replace('__', '_'), "")
agent_rank_statistics = arm_rank_statistics(agent_ranks)
diagnostic_tables(report, agent_rank_statistics, {'rank_statistics_by_arm': ['mean_clicked_rank', 'first_position_click_share']}, prefix='rq3_agents', context="", extra_ids=['arm'])

## Actual module use and reflection outcomes


In [ ]:
agent_diagnostics = label_conditions(results['agent'].loc[results['agent'].study.eq('movielens_feedback_a300_e804') & results['agent'].model.eq('agentic_recommender') & results['agent'].stage.str.startswith(('L2', 'L3'))])
modules = label_conditions(results['modules'].loc[results['modules'].study.eq('movielens_feedback_a300_e804') & results['modules'].model.eq('agentic_recommender') & results['modules'].stage.str.startswith(('L2', 'L3'))])
report.table(horizon_counts(modules, ['invocations', 'decisions_invoking'], extra_ids=['module']), 'rq3_module_totals', "")
diagnostic_tables(report, modules, {'module_rates': ['invocations', 'invocation_rate']}, prefix='rq3_agents', context="", extra_ids=['module'])

In [ ]:
report.table(reflection_totals(agent_diagnostics), 'rq3_reflection_totals', "")
diagnostic_tables(report, agent_diagnostics, {'reflection': ['reflection_requested', 'reflection_applied', 'reflection_failed'], 'reflection_failure': ['reflection_error_rate']}, prefix='rq3_agents', context="")
figure = trajectory_grid(agent_diagnostics, ['reflection_requested', 'reflection_error_rate'], title='Agent reflection use and failed-update rates', scales=scales, series='condition_label')
report.figure(figure, agent_diagnostics, 'rq3_reflection_diagnostics', "")

## Applied profile changes


In [ ]:
report.table(horizon_counts(agent_diagnostics, ['profile_changes']), 'rq3_profile_change_totals', "")
diagnostic_tables(report, agent_diagnostics, {'profile_changes': ['profile_changes', 'profile_change_rate', 'profile_changed_users'], 'distinct_changed_users': ['cumulative_profile_changed_users']}, prefix='rq3_agents', context="")
figure = trajectory_grid(agent_diagnostics, ['profile_changes', 'cumulative_profile_changed_users'], title='Applied profile updates and distinct changed users', scales=scales, series='condition_label')
report.figure(figure, agent_diagnostics, 'rq3_profile_updates', "")

## Seeded qualitative profile review


In [ ]:
review = results['qualitative_review']
report.table(review, 'rq3_profile_review_selection', "", columns=['seed', 'population_seed', 'user_id', 'review_order', 'eligible_users', 'applied_change_count', 'first_change_iteration', 'last_change_iteration', 'manual_assessment'])
profile_review_details(report, review, results['profile_changes'])

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['head_click_share', 'weighted_head_exposure']:
    table = label_ablation_comparisons(cross_seed_table(results, 'rq3', [metric]), include_order=False)
    report.table(table, f'rq3_seed_consistency_{metric}', "")